# Treinamento e avaliação - desafio CAEd

Este notebook apresenta as etapas e os resultados do pipeline canônico em `treinar.py`. O treinamento e a previsão do app usam um documento completo por exemplo; a estratégia anterior de trechos aparece apenas como baseline comparativo. As métricas abaixo foram atualizadas em 16/10/2026. A última célula repete o treinamento com o kernel da `.venv`.


## 1. Auditoria e divisão dos documentos

O dataset inicial tem 2.076 arquivos. Foram excluídos 24 documentos envolvidos em conflitos de rótulo e quatro duplicatas; não havia arquivos vazios. Os 2.048 documentos restantes são divididos por classe em treino, validação e teste antes de qualquer treinamento. Cada documento e seu hash ficam em apenas uma divisão.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RESULTADOS = Path("resultados")
divisao = pd.read_csv(RESULTADOS / "divisao_dados.csv")
vazios = pd.read_csv(RESULTADOS / "vazios.csv")
conflitos = pd.read_csv(RESULTADOS / "conflitos.csv")
duplicatas = pd.read_csv(RESULTADOS / "duplicatas.csv")
resumo_auditoria = pd.DataFrame({
    "Etapa": ["Arquivos originais", "Vazios", "Conflitos de rótulo", "Duplicatas", "Documentos válidos"],
    "Quantidade": [2076, len(vazios), len(conflitos), len(duplicatas), len(divisao)],
})
display(resumo_auditoria)
display(pd.crosstab(divisao["classe"], divisao["conjunto"]))


,Etapa,Quantidade
0,Arquivos originais,2076
1,Vazios,0
2,Conflitos de rótulo,24
3,Duplicatas,4
4,Documentos válidos,2048


conjunto,teste,treino,validacao
classe,,,
Ensino Fundamental I,44,206,44
Ensino Fundamental II,48,227,49
Ensino Médio,93,431,92
Ensino Superior,123,569,122


## 2. Classificação de documentos completos

Cada exemplo de treino é um documento completo, rotulado pela pasta de origem. O pipeline principal combina TF-IDF de palavras e pares de palavras com sequências de caracteres de três a cinco posições. Na validação, essa representação melhorou em relação à anterior, que incluía seis medidas numéricas; por isso, essas medidas não entram no novo classificador. Regressão logística e SVM linear são comparadas por F1 macro. O método anterior por trechos permanece apenas como referência comparativa.


In [ ]:
comparacao = pd.read_csv(RESULTADOS / "comparacao_validacao.csv")
display(comparacao)


                          modelo                             unidade  acuracia  f1_macro  f1_ponderado
0  documento_completo_svm_linear                  documento completo  0.947883  0.938300      0.947148
1   documento_completo_logistica                  documento completo  0.934853  0.922112      0.933700
2     metodo_anterior_svm_linear  votação de até 3 trechos/documento  0.899023  0.876994      0.898086
3      metodo_anterior_logistica  votação de até 3 trechos/documento  0.892508  0.876719      0.890472
4      referencia_mais_frequente                           documento  0.397394  0.142191      0.226024

## 3. Avaliação final no conjunto de teste

A tabela detalha as previsões do modelo selecionado, treinado em documentos completos. A comparação no teste usa os mesmos 308 documentos para o modelo atual e para a votação anterior entre trechos. Os rótulos são herdados das pastas do dataset, não de uma anotação independente da complexidade de cada texto.

A acurácia do modelo atual foi 93,51% e a F1 macro 0,9163. Dois textos externos enviados como exemplos — um esperado como Ensino Médio e outro como Ensino Superior — ainda foram previstos como Ensino Fundamental II. Eles não entraram no treinamento. A melhora no teste interno, portanto, não garante acerto em textos de outro estilo.


In [ ]:
import json
from processamento import CLASSES
metricas = json.loads((RESULTADOS / "metricas_teste.json").read_text(encoding="utf-8"))
matriz = pd.read_csv(RESULTADOS / "matriz_confusao.csv", index_col=0)
comparacao_teste = pd.read_csv(RESULTADOS / "comparacao_teste.csv")
linhas_metricas = []
for classe in CLASSES:
    m = metricas[classe]
    linhas_metricas.append({"Classe": classe, "Documentos": int(m["support"]), "Precisão": m["precision"], "Recall": m["recall"], "F1": m["f1-score"]})
linhas_metricas.append({"Classe": "Acurácia", "Documentos": int(metricas["weighted avg"]["support"]), "Precisão": None, "Recall": None, "F1": metricas["accuracy"]})
linhas_metricas.append({"Classe": "F1 macro", "Documentos": int(metricas["macro avg"]["support"]), "Precisão": metricas["macro avg"]["precision"], "Recall": metricas["macro avg"]["recall"], "F1": metricas["macro avg"]["f1-score"]})
tabela_metricas = pd.DataFrame(linhas_metricas)
display(tabela_metricas)
display(comparacao_teste)
display(matriz)


               Classe  Documentos  Precisão   Recall       F1
 Ensino Fundamental I          44  0.891304 0.931818 0.911111
Ensino Fundamental II          48  0.925000 0.770833 0.840909
         Ensino Médio          93  0.909091 0.967742 0.937500
      Ensino Superior         123  0.975610 0.975610 0.975610
             Acurácia         308       NaN      NaN 0.935065
             F1 macro         308  0.925251 0.911501 0.916282

                 modelo                            unidade  acuracia  f1_macro  f1_ponderado
     documento_completo                 documento completo  0.935065  0.916282      0.933896
metodo_anterior_trechos votação de até 3 trechos/documento  0.886364  0.863438      0.883856

                       Ensino Fundamental I  Ensino Fundamental II  Ensino Médio  Ensino Superior
real                                                                                             
Ensino Fundamental I                     41                      3             0                0
Ensino Fundamental II                     5                     37             6                0
Ensino Médio                              0                      0            90                3
Ensino Superior                           0                      0             3              120

## 4. Cobertura das mutações

O experimento usa 100 documentos completos da validação. A cobertura indica quantos documentos contêm ao menos uma palavra do vocabulário de mutação; a mudança de classe é comparada entre a previsão do documento original e a variante. Cada variante pode receber até duas substituições.


In [ ]:
experimentos = pd.read_csv(RESULTADOS / "experimentos_mutacao.csv")
reais = experimentos[experimentos["origem"] != "demonstracao"]
alterados = reais[reais["n_alteracoes"] > 0]
exemplos_mutacao = alterados[["origem", "original", "variante", "n_alteracoes", "mudou_classe"]].head(5).copy()
exemplos_mutacao["original"] = exemplos_mutacao["original"].str.slice(0, 120)
exemplos_mutacao["variante"] = exemplos_mutacao["variante"].str.slice(0, 120)
print(f"Cobertura: {len(alterados)}/{len(reais)} documentos ({len(alterados) / len(reais):.1%})")
print(f"Mudança de classe entre documentos alterados: {alterados['mudou_classe'].mean():.1%}")
display(exemplos_mutacao)


Cobertura: 100/100 documentos (100.0%)
Mudança de classe entre documentos alterados: 1.0%


                            origem                                                                                                                 original                                                                                                                 variante  n_alteracoes  mudou_classe
       3_Ensino_Medio/352_1713.txt E ainda o timbó, designação genérica de várias espécies de plantas com propriedades sedativas que eram utilizadas para p E ainda o timbó, designação genérica de várias espécies de plantas com propriedades sedativas que eram utilizadas para p             2         False
       3_Ensino_Medio/611_1972.txt Introdução à Biologia/História da Biologia/Do Renascimento à Idade Moderna O Renascimento na Europa veio renovar o inter Introdução à Biologia/História da Biologia/Do Renascimento à Idade Moderna O Renascimento na Europa veio renovar o inter             2         False
    4_Ensino_Superior/360_1392.txt Assim, dizemos que: Da mesma forma que: Seja a fun

## 5. Reexecutar o treinamento

Esta célula chama a implementação usada por `python treinar.py`. Ela atualiza o modelo e os arquivos em `resultados/`. Se o kernel foi aberto antes da edição dos módulos, o recarregamento abaixo garante o uso do código atual. O treinamento também atualiza a comparação entre documentos completos e a estratégia anterior.


In [ ]:
import importlib
import processamento
import mutacoes
import treinar

importlib.reload(processamento)
importlib.reload(mutacoes)
treinar = importlib.reload(treinar)
resultado = treinar.executar_treinamento()
